# TikTok BNPL corpus: collection

Collects public TikTok videos about Buy Now, Pay Later (BNPL) from accounts registered in Italy, the public profiles of selected creators, and comments on selected videos, through the TikTok Research API. Raw data is written to `data/raw/tiktok/`, which Git ignores. Every step is resumable: when the daily quota runs out, rerun the notebook the next day and it continues where it stopped.

Credentials are read from the `TIKTOK_CLIENT_KEY` and `TIKTOK_CLIENT_SECRET` environment variables, then from the ignored `.env` file in the project root, or typed at the prompt. They are never stored in the notebook. This notebook displays counts only, never text or usernames.

In [1]:
import logging
import sys
from pathlib import Path

from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'scripts').is_dir(): PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
logging.basicConfig(level=logging.WARNING, format='%(levelname)s %(name)s: %(message)s')
from datetime import date, timedelta
from getpass import getpass

import pandas as pd

from scripts import tiktok_analysis as analysis
from scripts import tiktok_collection as collection
from scripts import tiktok_terms as terms
from scripts.tiktok_client import TikTokClient, read_credentials

RAW_DIR = collection.RAW_TIKTOK
PILOT_DIR = RAW_DIR / 'pilot'
PILOT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Authenticate

In [2]:
credentials = read_credentials(PROJECT_ROOT / '.env') or (
    getpass('Client key: '), getpass('Client secret: ')
)
client = TikTokClient(*credentials)

## 2. Pilot the search terms

Each candidate keyword and hashtag was queried alone over the last complete 30 days (one request per term) to measure its yield and read a sample of its videos. The pilot runs only when its files are missing; samples stay in the ignored raw folder because they contain text and usernames.

In [3]:
yield_path, sample_path = PILOT_DIR / 'term_yield.csv', PILOT_DIR / 'term_samples.csv'
if not yield_path.exists():
    pilot_end = date.today() - timedelta(days=collection.INDEXING_LAG_DAYS + 1)
    pilot_yield, pilot_samples = collection.run_pilot(
        client, terms.CANDIDATE_KEYWORDS, terms.CANDIDATE_HASHTAGS,
        pilot_end - timedelta(days=29), pilot_end,
    )
    pilot_yield.to_csv(yield_path, index=False)
    pilot_samples.to_csv(sample_path, index=False, encoding='utf-8-sig')
pilot_yield = pd.read_csv(yield_path)
pilot_yield['kept'] = ~pilot_yield['term'].isin(terms.DROPPED_TERMS) | (
    pilot_yield['field'] == 'hashtag_name'
)
display(pilot_yield)

,field,term,first_page_videos,has_more,kept
0,keyword,klarna,98,True,True
1,keyword,scalapay,97,True,True
2,keyword,heylight,98,True,True
3,keyword,pagolight,10,False,True
4,keyword,soisy,1,False,False
5,keyword,pagodil,15,False,True
6,keyword,paga in 3 rate,99,True,True
7,keyword,pagare a rate,99,True,False
8,keyword,pagamento a rate,94,True,False
9,keyword,rate senza interessi,98,True,True


**Reading the pilot.** Provider names, *paga in 3 rate* and *rate senza interessi* fill a whole first page (about 100 videos) in 30 days, so they drive the corpus. The English and generic BNPL phrases return only a handful of videos, but those few are disproportionately commentary and first-person accounts, so they are kept. Reading the samples showed three problems, recorded in `scripts/tiktok_terms.py`:

- *soisy* returned a single unrelated video.
- *pagare a rate* and *pagamento a rate* mostly returned debt repayment, tax installments, and generic retailer financing rather than BNPL.
- Even for the kept terms, most videos are shop and service ads that list Klarna or Scalapay as a payment option. The analysis therefore separates ads from commentary and personal experience instead of treating every video as discourse about BNPL.

## 3. Freeze the query and collect videos

The query is: accounts registered in Italy **and** any kept keyword **or** hashtag. It is frozen in `data/raw/tiktok/query_config.json` on the first run, and the collection refuses to continue with a different query. Windows are 7 days long; the most recent days are skipped until TikTok has indexed them.

In [4]:
query = collection.build_query(terms.FINAL_KEYWORDS, terms.FINAL_HASHTAGS)
windows = collection.date_windows(terms.COLLECTION_START, terms.COLLECTION_END)
print(f'{len(windows)} complete windows from {windows[0][0]} to {windows[-1][1]}')
video_status = collection.collect_videos(client, query, windows, RAW_DIR)
display(video_status['status'].value_counts().rename('windows').to_frame())
print('Requests this session:', client.request_count)

146 complete windows from 2024-01-01 to 2026-10-18


WARNING scripts.tiktok_collection: Quota exhausted at window 20260914_20260920: /v2/research/video/query/ failed (429, daily_quota_limit_exceeded): The API daily quota limit exceeded. Please try again later [log_id=20261026113747E1AA2B875621D623BF7D]


,windows
status,
done,141
pending,5


Requests this session: 1


## 4. Creator profiles

Each profile costs one request, so profiles are fetched for every creator of a non-ad post plus the 300 most active creators overall. Private or deleted accounts are stored as `found = False`.

In [5]:
raw = analysis.load_raw(RAW_DIR)
salt = analysis.pseudonym_salt(RAW_DIR)
creators = analysis.build_creator_table(raw['videos'], raw['users'], salt)
corpus, funnel = analysis.build_video_corpus(raw['videos'], creators)
wanted = analysis.select_profile_creators(corpus, top_active=300)
print(f'{len(wanted)} creators selected for profiles')
users = collection.collect_users(client, wanted, RAW_DIR)
print(f"{len(users)} profiles stored, {int(users['found'].sum())} public")

2348 creators selected for profiles


WARNING scripts.tiktok_collection: Quota exhausted while fetching profiles: /v2/research/user/info/ failed (429, daily_quota_limit_exceeded): The API daily quota limit exceeded. Please try again later [log_id=202610261140055AE857C1BF638A453F83]


997 profiles stored, 996 public


## 5. Comments on selected videos

Comments are collected for the most-commented videos of each post type: up to 300 personal-experience videos, 300 commentary videos, 100 unclassified posts, and 100 shop ads. The weighting favours the posts where people discuss BNPL rather than ask about a product.

In [6]:
creators = analysis.build_creator_table(raw['videos'], users, salt)
corpus, funnel = analysis.build_video_corpus(raw['videos'], creators)
selected = analysis.select_comment_videos(corpus)
print(f'{len(selected)} videos selected for comments')
comment_status = collection.collect_comments(client, selected, RAW_DIR)
display(comment_status['status'].value_counts().rename('videos').to_frame())
print('Requests this session:', client.request_count)

309 videos selected for comments


,videos
status,
done,309


Requests this session: 2


When no window or comment thread is `pending`, collection is complete. Continue with `tiktok_analysis.ipynb`.